In [1]:
import os
os.environ ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
pip install chromadb

In [2]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceHubEmbeddings,HuggingFaceInstructEmbeddings
from langchain.llms import HuggingFaceHub
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI
from langchain.llms.base import LLM
import time
import numpy as np
from typing import List
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sentence_transformers.quantization import quantize_embeddings

/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN']=''
os.environ['OPENAI_API_KEY'] =''

In [5]:
from mixedbread_ai.client import MixedbreadAI


In [6]:
mxbai = MixedbreadAI(api_key="")
model_name = ""
persist_directory=''

In [7]:
class MyEmbeddings:
    def __init__(self):
        self.model =SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

        
    # def get_embeddings(self, texts: List[str], model: str, prompt: str = None) -> np.ndarray:
    #     res = mxbai.embeddings(
    #         input=texts,
    #         model=model,
    #         prompt=prompt
    #     )
    #     embeddings = [entry.embedding for entry in res.data]
    #     return np.array(embeddings)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # return self.get_embeddings(
        #     texts,
        #     model_name,
        #     "Represent these documents for searching relevant sentences"
        # )
        return [self.model.encode(t).tolist() for t in texts]
    
    def embed_query(self, text:str) -> List[float]:
        return self.model.encode(text).tolist()
    
    

In [8]:
embedding=MyEmbeddings()

In [ ]:
# Load all text files from the specified directory manually
directory_path = ''
doc_texts = []

for filename in os.listdir(directory_path):
    filepath = os.path.join(directory_path, filename)

    # Check if the file is a text file (you can adjust this condition as needed)
    if filename.endswith('.txt'):
        with open(filepath, 'r', encoding='utf-8') as file:
            content = file.read()  # Read the entire content as text
            # Create a Document object manually
            document = Document(page_content=content, metadata={"source": filepath})
            doc_texts.append(document)
    else:
        print(f"Skipping unsupported file: {filename}")


character_text_splitter=CharacterTextSplitter(chunk_size=5000,chunk_overlap=150)
doc_chunks=character_text_splitter.split_documents(doc_texts)
vectordb=Chroma.from_documents(documents=doc_chunks, embedding=embedding, persist_directory=persist_directory)

In [10]:
vectordb=Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [ ]:
vectordb=Chroma.from_documents(documents=doc_texts, embedding=embedding, persist_directory=persist_directory)